# Experiment 3 — Speech ↔ Text CKA Alignment

*Paper Sec 2.5. Reports CKA + random-permutation Δ.*

For **MIMI** and **MIMO** we extract:

* Speech-side: the final accumulated layer of the codec for each MFA-aligned word, time-pooled to a single vector.
* Text-side: the LLM input embedding of the same word, averaged over its sub-tokens.

We compute linear CKA between the two ``[N, d]`` matrices and contrast against a 100-permutation baseline that shuffles the row alignment.

Paper numbers: MIMI CKA = 0.329 (Δ = 0.087); MIMO CKA = 0.122 (Δ = 0.054).

In [ ]:
import os, sys, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Load word DataFrame

In [ ]:
WORDPAIRS_DIR = os.path.join(REPO_ROOT, 'data', 'word_pairs')
df = pd.read_pickle(os.path.join(WORDPAIRS_DIR, 'librispeech.df.pkl'))
from src.data import resolve_audio_paths
LIBRISPEECH_DIR = os.environ.get('LIBRISPEECH_DIR', os.path.join(REPO_ROOT, 'data', 'LibriSpeech'))
df = resolve_audio_paths(df, LIBRISPEECH_DIR)
_missing = [p for p in df.path.unique() if not os.path.exists(p)]
if _missing:
    print(f'WARNING: {len(_missing)}/{df.path.nunique()} audio files not found under {LIBRISPEECH_DIR}; skipping those rows')
    df = df[df.path.map(os.path.exists)].reset_index(drop=True)
# Subset to one occurrence per unique word — CKA needs paired rows
df_words = df.groupby('text').first().reset_index()
print(f'{len(df_words):,} unique words')
N_WORDS = 500  # raise once everything works
df_words = df_words.sample(min(N_WORDS, len(df_words)), random_state=0).reset_index(drop=True)
df_words.head(3)

## 2. Speech-side: codec features (final accumulated layer, time-pooled)

In [ ]:
from src.codecs import load_mimi, load_mimo
from src.analysis import extract_codec_features

def speech_matrix(codec, df):
    df_feat = extract_codec_features(codec, df, pool='mean')
    return np.stack([f[-1] for f in df_feat.feat.tolist()])  # [N, D] from last layer

## 3. Text-side: LLM input embedding averaged over sub-tokens

The LLM checkpoint must match the codec it pairs with. The paper uses **Moshi** for MIMI and the **MiMo-Audio** LLM for MIMO.

In [ ]:
from transformers import AutoTokenizer, AutoModel

def text_matrix(words, llm_name):
    tok = AutoTokenizer.from_pretrained(llm_name)
    model = AutoModel.from_pretrained(llm_name, torch_dtype=torch.float32, device_map={'': 'cpu'})
    emb = model.get_input_embeddings()
    out = []
    for w in words:
        ids = tok(w, return_tensors='pt', add_special_tokens=False)['input_ids'][0]
        with torch.no_grad():
            out.append(emb(ids).mean(dim=0).cpu().numpy())
    return np.stack(out)

TEXT_MODELS = {
    'mimi': 'kyutai/moshiko-pytorch-bf16',     # verify this is the LLM tied to MIMI in your setup
    'mimo': 'XiaomiMiMo/MiMo-Audio-7B-Base',   # verify this is the LLM tied to MIMO in your setup
}

## 4. CKA + permutation baseline

In [ ]:
from src.analysis import cka_with_baseline

results = {}
for name, factory in [
    ('mimi', lambda: load_mimi(device=DEVICE)),
    ('mimo', lambda: load_mimo(checkpoint=os.environ.get('MIMO_CHECKPOINT', '<path-to-MiMo-Audio-Tokenizer>'), device=DEVICE)),
]:
    try:
        codec = factory()
    except Exception as e:
        print(f'skipping {name}: {e}'); continue
    S = speech_matrix(codec, df_words)
    T = text_matrix(df_words.text.tolist(), TEXT_MODELS[name])
    r = cka_with_baseline(S, T, n_permutations=100, seed=0)
    results[name] = r
    print(f'{name}: CKA={r["cka"]:.3f}  baseline={r["baseline_mean"]:.3f}  Δ={r["delta"]:.3f}')
    del codec; torch.cuda.empty_cache()

In [ ]:
out_path = os.path.join(REPO_ROOT, 'data', 'cka_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(results, f)
print('wrote', out_path)